# 02 - Preprocessing and Feature Engineering
## Cleaning with a logged audit trail, then the cached analytical dataset

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

Run the four cleaning steps with a row count logged after each one, derive the analysis features
once, and write the result to Parquet. Every notebook from 03 onward reads that Parquet instead
of reparsing the CSV, which is both faster and guarantees they all analyse identical rows.

The cleaning and feature rules live in `da_common.py` so they are defined in exactly one place.


In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 02 - Preprocessing and Feature Engineering")
spark = get_spark("02 preprocessing")

### 1. Cleaning pipeline

1. Missing-value sentinels become real nulls.
2. The Unix-millisecond timestamp is parsed and calendar features are derived.
3. Ratings are range-checked to the valid 1 to 5 band.
4. Duplicate reviews (same user, product, timestamp) are dropped and empty bodies removed. A
   missing title is filled with an empty string rather than costing the whole row.

In [ ]:
raw = read_raw(spark).cache()
print(f"raw rows: {raw.count():,}\n")

df, audit = preprocess(raw)

In [ ]:
print(f"Retained {audit['rows'].iloc[-1]:,} of {audit['rows'].iloc[0]:,} rows "
      f"({audit['retained_pct'].iloc[-1]:.2f}%)")
save_table(audit, "tbl07_preprocessing_audit")
audit

### 2. Feature engineering

Length and word count come from the text, a log-scaled helpful-vote column and a binary
high-helpfulness flag come from the vote count, and product-level aggregates are computed once
and joined back so later notebooks never recompute them.

In [ ]:
df = engineer(df)
print(f"columns ({len(df.columns)}):", ", ".join(df.columns))
df.select("Category", "rating", "verified_purchase", "helpful_vote", "review_length",
          "review_word_count", "log_helpful_vote", "is_highly_helpful", "review_year",
          "product_avg_rating", "product_review_count").show(5)

### 3. Persist the analytical dataset

In [ ]:
(df.drop("images").write.mode("overwrite").parquet(CLEAN_PARQUET))
print("written ->", CLEAN_PARQUET)

check = spark.read.parquet(CLEAN_PARQUET)
n_clean = check.count()
print(f"read-back verification: {n_clean:,} rows x {len(check.columns)} columns")
assert n_clean == audit["rows"].iloc[-1], "row count changed on write, investigate"
print("row count matches the audit trail.")

save_table(pd.DataFrame([{"clean_rows": n_clean,
                          "raw_rows": int(audit["rows"].iloc[0]),
                          "retained_pct": float(audit["retained_pct"].iloc[-1])}]),
           "tbl08_clean_dataset_summary");

### Findings

Cleaning removes only a fraction of a percent of the extract, and the audit trail shows exactly
which step removed what, so no row disappears without a recorded reason. The written Parquet is
the single input for notebooks 03 through 08.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")